# Modelos de SentencePiece

In [8]:
!pip install datasets

In [9]:
!pip install sentencepiece
#!wget https://huggingface.co/datasets/large_spanish_corpus

In [10]:
from datasets import load_dataset
# ⚠️ Esto descargará TODO el dataset (puede tardar y usar mucho disco)
all_wiki = load_dataset("jhonparra18/spanish_billion_words_clean", split="train").take(500000)
print(len(all_wiki))  # Ahora sí funciona

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00018.parquet:   0%|          | 0.00/212M [00:00<?, ?B/s]

data/train-00001-of-00018.parquet:   0%|          | 0.00/229M [00:00<?, ?B/s]

data/train-00002-of-00018.parquet:   0%|          | 0.00/240M [00:00<?, ?B/s]

... (14 outputs left out)

data/train-00017-of-00018.parquet:   0%|          | 0.00/220M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/46925295 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/18 [00:00<?, ?it/s]

500000


In [11]:
all_wiki


Dataset({
    features: ['text'],
    num_rows: 500000
})

In [12]:
import os


try:
    os.mkdir('./wiki2')
except FileExistsError:
    print('The directory already exists')

In [13]:
from tqdm.auto import tqdm  # for our loading bar

text_data = []
file_count = 0

for sample in tqdm(all_wiki):
    # remove newline characters from each sample as we need to use exclusively as seperators
    sample = sample['text'].replace('\n', '\s')
    text_data.append(sample)
    if len(text_data) == 5_000:
        # once we hit the 5K mark, save to file
        with open(f'./wiki2/text_{file_count}.txt', 'w', encoding='utf-8') as fp:
            fp.write('\n'.join(text_data))
        text_data = []
        file_count += 1
# after saving in 5K chunks, we may have leftover samples, we save those now too
with open(f'./wiki2/text_{file_count}.txt', 'w', encoding='utf-8') as fp:
    fp.write('\n'.join(text_data))

<>:8: SyntaxWarning: invalid escape sequence '\s'
<>:8: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_1098/3745086507.py:8: SyntaxWarning: invalid escape sequence '\s'
  sample = sample['text'].replace('\n', '\s')


  0%|          | 0/500000 [00:00<?, ?it/s]

In [14]:
from pathlib import Path
paths = [str(x) for x in Path('./wiki2').glob('**/*.txt')]



In [15]:
len(paths)

101

# Modelo básico de prefijos y decodificadores

Sentencepiece se entrena  con un conjunto grande de 683 archivos de texto con un vocabulario de 10000 tokens sobre un modelo de prefijos. Este conjunto de archivos proviene del dataset de [JRC](https://opus.nlpl.eu/JRC-Acquis.php) de  [large spanish corpus]( https://huggingface.co/datasets/large_spanish_corpus) como una parte  de uno de los corpus más grande del español.

In [16]:
#@title Texto de título predeterminado
import glob
import sentencepiece as spm
spm.SentencePieceTrainer.Train(input=glob.glob('wiki2/*.txt'), vocab_size=10000,model_prefix='m')
#Una vez completado el entrenamiento, se generan los archivos m.model y m.vocab.
#Estos 10000 tokens únicos pueden ser palabras completas, subpalabras o caracteres individuales, dependiendo del algoritmo utilizado.
sp = spm.SentencePieceProcessor()
sp.Load("m.model")
print(sp.encode_as_pieces('Todos los bachilleres quieren estudiar en Univalle'))
print(sp.encode_as_ids('Todos los bachilleres quieren estudiar en Univalle'))
#sp.EncodeAsIds("Todos los bachilleres quieren estudiar en Univalle")

#spm.SentencePieceTrainer.Train(input=glob.glob('f1/*/*.txt'), vocab_size=10000,model_prefix='m')

['▁', 'T', 'o', 'dos', '▁los', '▁bachiller', 'es', '▁quieren', '▁estudiar', '▁en', '▁', 'U', 'n', 'iva', 'lle']
[146, 0, 47, 113, 12, 8931, 33, 3676, 6466, 8, 146, 0, 22, 2376, 2255]


In [17]:
print(sp.encode_as_pieces('Todos los bachilleres quieren estudiar en Univalle'))
print(sp.encode_as_ids('Todos los bachilleres quieren estudiar en Univalle'))
print(sp.decode_ids([686, 29, 13, 981, 5, 70, 32, 233, 41, 143, 89, 48, 135, 124, 15]))


['▁', 'T', 'o', 'dos', '▁los', '▁bachiller', 'es', '▁quieren', '▁estudiar', '▁en', '▁', 'U', 'n', 'iva', 'lle']
[146, 0, 47, 113, 12, 8931, 33, 3676, 6466, 8, 146, 0, 22, 2376, 2255]
nes como con estándars entre lo vistas está otros hastat ya bar del


# SentencePiece con el modelo bpe real


SentencePiece soporta el modelo original de BPE para segmentación de subpalabras

In [18]:
import sentencepiece as spm
spm.SentencePieceTrainer.train(input=glob.glob('wiki2/*.txt'),vocab_size=2000, model_prefix='bpe')
sp_bpe = spm.SentencePieceProcessor()
sp_bpe.load('bpe.model')

True

En este caso se comrpueba que los ids de los dos modelos el de palabra y el bpe son diferentes

In [19]:
print('*** BPE ***')
print(sp_bpe.encode_as_pieces('Todos los bachilleres quieren estudiar en Univalle'))
print(sp_bpe.encode_as_ids('Todos los bachilleres quieren estudiar en Univalle'))
#print(sp_word.encode_as_pieces('Este es el librito de Job'))
#print(sp_word.encode_as_ids('Este es el librito de Job'))

*** BPE ***
['▁', 'T', 'o', 'dos', '▁los', '▁ba', 'chi', 'lle', 'res', '▁quiere', 'n', '▁estudia', 'r', '▁en', '▁', 'U', 'n', 'iva', 'lle']
[55, 0, 34, 123, 15, 167, 1112, 304, 264, 1203, 12, 1438, 11, 8, 55, 0, 12, 837, 304]


# Segmentación de palabras con SentencePiece

In [20]:
import sentencepiece as spm
spm.SentencePieceTrainer.train(input=glob.glob('wiki2/*.txt'),
    vocab_size=20000,
    model_prefix='word_es',
    model_type='bpe',
    character_coverage=1.0,
)
sp_word = spm.SentencePieceProcessor()
# Carga y uso
sp = spm.SentencePieceProcessor()
sp.load('word_es.model')

# ¡Normaliza a minúsculas!
text = "Todos los bachilleres quieren estudiar en Univalle"
text_norm = text.lower()

pieces = sp.encode_as_pieces(text_norm)
ids = sp.encode_as_ids(text_norm)

print("Tokens:", pieces)
print("IDs:", ids)

Tokens: ['▁todos', '▁los', '▁bachiller', 'es', '▁quieren', '▁estudiar', '▁en', '▁un', 'ival', 'le']
IDs: [422, 62, 14540, 11, 5660, 9808, 28, 46, 4051, 60]


# Que puedo haber pasado con _profesores_estudian

# Modelo de prefijos  con SentencePiece

In [21]:
import sentencepiece as spm
spm.SentencePieceTrainer.train(input=glob.glob('wiki2/*.txt'),vocab_size=2000,model_prefix='m')

# makes segmenter instance and loads the model file (m.model)
sp = spm.SentencePieceProcessor()
sp.load('m.model')
print(sp.encode_as_pieces('Todos los bachilleres quieren estudiar en Univalle'))
#print(sp_word.encode_as_pieces('¿Quieres saber cómo hacer una aplicación para Android Wear? Te lo explicamos &gt; http://t.co/IjgCIkigvm http://t.co/tUZhU3wrhx '))
print(sp.encode_as_ids('Todos los bachilleres quieren estudiar en Univalle'))

['▁', 'T', 'o', 'dos', '▁los', '▁ba', 'chi', 'lle', 'res', '▁quiere', 'n', '▁estudia', 'r', '▁en', '▁', 'U', 'n', 'iva', 'lle']
[55, 0, 34, 123, 15, 167, 1112, 304, 264, 1203, 12, 1438, 11, 8, 55, 0, 12, 837, 304]


# Modelo caracter con SentencePiece

In [22]:
spm.SentencePieceTrainer.train(input=glob.glob('wiki2/*.txt'),vocab_size=2000,model_prefix='char')

sp_char = spm.SentencePieceProcessor()
sp_char.load('char.model')

print(sp_char.encode_as_pieces('Todos los bachilleres quieren estudiar en Univalle'))
print(sp_char.encode_as_ids('Todos los bachilleres quieren estudiar en Univalle'))

['▁', 'T', 'o', 'dos', '▁los', '▁ba', 'chi', 'lle', 'res', '▁quiere', 'n', '▁estudia', 'r', '▁en', '▁', 'U', 'n', 'iva', 'lle']
[55, 0, 34, 123, 15, 167, 1112, 304, 264, 1203, 12, 1438, 11, 8, 55, 0, 12, 837, 304]


In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Modelo de usuario defindo para usar con BERT

In [24]:
spm.SentencePieceTrainer.train('--input=/content/drive/MyDrive/PLN-2026-I/Tokenizadores/tweets_clean.txt --model_prefix=m_user --user_defined_symbols=<sep>,<cls> --vocab_size=2000')
# ids are reserved in both mode.
# <unk>=0, <s>=1, </s>=2, <sep>=3, <cls>=4
# user defined symbols allow these symbol to apper in the text.
sp_user = spm.SentencePieceProcessor()
sp_user.load('m_user.model')
print(sp_user.encode_as_pieces('this<sep> is a test hello world<cls>'))
print(sp_user.piece_to_id('<sep>'))  # 3
print(sp_user.piece_to_id('<cls>'))  # 4
print('3=', sp_user.decode_ids([3]))  # decoded to <sep>
print('4=', sp_user.decode_ids([4]))  # decoded to <cls>
print(sp_user.encode_as_pieces('<s>Tw33t<sep> # @dude_really<unk> #hash_tag $hit (g@y) retard#d @dude. 😀😀 !😀abc %😀lol #hateit #hate.it $%&/ f*ck- in love_twitter<cls></s>'))
#print('0=', sp_user.decode_ids([0]))  # decoded to <cls>

['▁', 't', 'hi', 's', '<sep>', '▁', 'is', '▁a', '▁te', 's', 't', '▁he', 'llo', '▁', 'w', 'or', 'l', 'd', '<cls>']
3
4
3= <sep>
4= <cls>
['▁', '<', 's', '>', 'T', 'w', '3', '3', 't', '<sep>', '▁#', '▁@', 'du', 'de', '_', 'rea', 'll', 'y', '<', 'un', 'k', '>', '▁#', 'ha', 's', 'h', '_', 't', 'a', 'g', '▁', '$', 'hi', 't', '▁(', 'g', '@', 'y', ')', '▁re', 't', 'ar', 'd', '#', 'd', '▁@', 'du', 'de', '.', '▁', '😀😀', '▁', '!', '😀', 'a', 'b', 'c', '▁', '%', '😀', 'lo', 'l', '▁#', 'ha', 'te', 'i', 't', '▁#', 'ha', 'te', '.', 'i', 't', '▁', '$', '%', '&', '/', '▁f', '*', 'ck', '-', '▁in', '▁lo', 've', '_', 't', 'wi', 't', 't', 'er', '<cls>', '<', '/', 's', '>']


In [25]:
import sentencepiece as spm
spm.SentencePieceTrainer.train('--input=/content/drive/MyDrive/PLN-2026-I/Tokenizadores/tweets_clean.txt --model_prefix=m_bos_as_user --user_defined_symbols=<s>,</s> --vocab_size=2000')

sp = spm.SentencePieceProcessor()
#Este modelo está entrenado con el modelo básico de sentencepiece que no tiene en cuenta los símbolos definidos
sp.load('m.model')
print(sp.encode_as_pieces('<s> Todos los colmbianos cantan</s>'))   # <s>,</s> are segmented. (default behavior)
print(sp.encode_as_pieces('<s>[INST] ¿Qué es el chunking? [/INST] El chunking es una técnica utilizada en NLP para dividir un texto en fragmentos significativos. </s>'))

sp = spm.SentencePieceProcessor()
sp.load('m_bos_as_user.model')
print(sp.encode_as_pieces('<s> Todos los colombianos cantan</s>'))   # <s>,</s> are handled as one token.

['▁', '<', 's', '>', '▁', 'T', 'o', 'dos', '▁los', '▁col', 'mb', 'ia', 'nos', '▁can', 'ta', 'n', '</', 's', '>']
['▁', '<', 's', '>[INST]', '▁', '¿Q', 'u', 'é', '▁es', '▁el', '▁', 'ch', 'un', 'k', 'ing', '?', '▁', '[/INST]', '▁', 'E', 'l', '▁', 'ch', 'un', 'k', 'ing', '▁es', '▁una', '▁t', 'é', 'c', 'n', 'ica', '▁utiliza', 'da', '▁en', '▁', 'NLP', '▁para', '▁di', 'vi', 'di', 'r', '▁un', '▁texto', '▁en', '▁f', 'ra', 'g', 'mento', 's', '▁significa', 'tivo', 's', '.', '▁', '</', 's', '>']
['▁', '<s>', '▁Todo', 's', '▁los', '▁', 'co', 'lo', 'mb', 'iano', 's', '▁can', 't', 'an', '</s>']


# Este entrena   un modelo de palabra con tweets


In [26]:
import sentencepiece as spm
spm.SentencePieceTrainer.train('--input=/content/drive/MyDrive/PLN-2026-I/Tokenizadores/tweets_clean.txt --model_prefix=m_word --model_type=word --vocab_size=2000')

sp_word = spm.SentencePieceProcessor()
sp_word.load('m_word.model')

print(sp_word.encode_as_pieces('¿Quieres<sep> saber cómo hacer una aplicación para Android Wear? Te lo explicamos &gt; http://t.co/IjgCIkigvm http://t.co/tUZhU3wrhx '))
print(sp_word.encode_as_ids('Este es el libro de Job.'))
print(sp_word.encode_as_pieces("@HyperKynetic si me los traes a la te querría para siempre 😁 que  en #ayunas por sacarme sangre :( "))
print(sp_word.encode_as_pieces('Tw33t # @dude_really #hash_tag $hit (g@y) retard#d @dude. 😀😀 !😀abc %😀lol #hateit #hate.it $%&/ f*ck- in love_twitter'))
print(sp_word.encode_as_pieces('Cigarette smoke contains poisonous elements that go into our body and make us suffer from it,due to which a deadly disease like cancer is born. Stop smoking tobacco! Please do listen to satsang daily on #GodMorningFriday#, #Coronvirus'))

['▁¿Quieres<sep>', '▁saber', '▁cómo', '▁hacer', '▁una', '▁aplicación', '▁para', '▁Android', '▁Wear?', '▁Te', '▁lo', '▁explicamos', '▁&gt;', '▁http://t.co/IjgCIkigvm▁http://t.co/tUZhU3wrhx']
[102, 15, 7, 785, 3, 0]
['▁@HyperKynetic', '▁si', '▁me', '▁los', '▁traes', '▁a', '▁la', '▁te', '▁querría', '▁para', '▁siempre', '▁😁', '▁que', '▁en', '▁#ayunas', '▁por', '▁sacarme', '▁sangre', '▁:(']
['▁Tw33t▁#▁@dude_really▁#hash_tag▁$hit▁(g@y)▁retard#d▁@dude.▁😀😀▁!😀abc▁%😀lol▁#hateit▁#hate.it▁$%&/▁f*ck-', '▁in', '▁love_twitter']
['▁Cigarette▁smoke▁contains▁poisonous▁elements▁that▁go▁into▁our▁body', '▁and', '▁make▁us▁suffer▁from▁it,due', '▁to', '▁which', '▁a', '▁deadly▁disease▁like▁cancer', '▁is', '▁born.▁Stop▁smoking▁tobacco!▁Please▁do▁listen', '▁to', '▁satsang▁daily▁on▁#GodMorningFriday#,▁#Coronvirus']
